# Test 2: Regresión lineal para estimación de SWF

## Dataset original

In [ ]:
# Import necessary libraries
import os
import sys
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cartopy.crs as ccrs
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import xarray as xr
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from CIMR.SurfaceWaterFraction_ATBD_main.algorithm.processing import colormaps
from CIMR.SurfaceWaterFraction_ATBD_main.algorithm.processing.validation_data_processing import load_lut, unravel_freqpol, atmospheric_corrections

# Open data
data_dir = os.path.join(os.getcwd(), '..', 'data')

# Load the LUT with reference land emissivities
lut_h_path = "CIMR/SurfaceWaterFraction_ATBD_main/data/lut_de_lannoy_K_h.csv"
lut_v_path = "CIMR/SurfaceWaterFraction_ATBD_main/data/lut_de_lannoy_K_v.csv"

# Try to open local dataset first, it's faster:
local_tds = "CIMR/SurfaceWaterFraction_ATBD_main/data/remote_data/CIMR_SWF_TDS_JNB_v3.nc"

tds_base = xr.open_dataset(local_tds)

# NOTE: we need to normalize VOD between 0 and 1 from original LPDR's values: [0-3] Neper
min_val = 0
max_val = 3
old_attrs = tds_base.VOD.attrs
tds_base["VOD"] = (tds_base.VOD - min_val) /(max_val - min_val)

old_attrs.update(
    {
        "Valid_range": "1-0",
        "Description": "NORMALIZED VOD, original is 0-3 Neper", "Units": None
    }
)
tds_base.VOD.attrs = old_attrs

In [ ]:
df_base = tds_base.to_dataframe().reset_index()

df_base = df_base.dropna(subset=['fwns'])

coords = ['lat', 'lon']   # cámbialas por tus coordenadas reales
X_base = df_base.drop(columns=['fwns'] + coords)
X_base = X_base.dropna(axis=1, how='all')
mask = X_base.notnull().all(axis=1)
X_base = X_base[mask]

y_base = df_base['fwns']
y_base = y_base[mask]

X_base_train, X_base_test, y_base_train, y_base_test = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

In [38]:
model_base = LinearRegression()
model_base.fit(X_base_train, y_base_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [39]:
y_base_pred = model_base.predict(X_base_test)
print("R2:", r2_score(y_base_test, y_base_pred))

R2: 0.7920319245710115


## Dataset procesado

In [40]:
tds_procesado = tds_base.copy()

tds_procesado = atmospheric_corrections(tds_procesado)

tds_procesado = load_lut(tds_procesado, lut_filepath=lut_h_path)
tds_procesado = load_lut(tds_procesado, lut_filepath=lut_v_path)

# unravel tbtoa layer into frequency and polarization arrays
tds_procesado = unravel_freqpol(tds_procesado, dvars=[
    "tbtoa", "tbboa_de_lannoy"
])

# Drop the multy dimentional dvars that we dont need anymore
tds_procesado = tds_procesado.drop_vars(["tbtoa", "tbboa_de_lannoy"])

# Get emissivity from TDS, Which has ERA5 skin temperature
for freq in ["19", "37"]:
    for pol in ["H", "V"]:
        tds_procesado[f"emiss{freq}{pol}_de_lannoy"] = tds_procesado[f"tbboa_de_lannoy{freq}{pol}"] / tds_procesado["surtep_ERA5"]

# Use IGBP to remove the ocean
tds_procesado = tds_procesado.where(tds_procesado.IGBP_landcover > 0)

In [41]:
df_procesado = tds_procesado.to_dataframe().reset_index()

df_procesado = df_procesado.dropna(subset=['fwns'])

coords = ['lat', 'lon']   # cámbialas por tus coordenadas reales
X_procesado = df_procesado.drop(columns=['fwns'] + coords)
X_procesado = X_procesado.dropna(axis=1, how='all')
mask = X_procesado.notnull().all(axis=1)
X_procesado = X_procesado[mask]

y_procesado = df_procesado['fwns']
y_procesado = y_procesado[mask]

X_procesado_train, X_procesado_test, y_procesado_train, y_procesado_test = train_test_split(
    X_procesado, y_procesado, test_size=0.2, random_state=42
)

In [42]:
model_procesado = LinearRegression()
model_procesado.fit(X_procesado_train, y_procesado_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [43]:
y_procesado_pred = model_procesado.predict(X_procesado_test)
print("R2:", r2_score(y_procesado_test, y_procesado_pred))

R2: 0.8635192600991222


## Dataset con ingeniería de características

In [ ]:
tds_ingenieria = tds_procesado.copy()

# SWF computation
ref_water_emiss_h = 0.288760

tds_ingenieria["Caracteristica1"] = (1) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["Caracteristica2"] = (tds_ingenieria.ref_land_emis_de_lannoy_K_h) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
tds_ingenieria["Caracteristica3"] = (tds_ingenieria.emiss19H_de_lannoy) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)
#tds_ingenieria["SWF_calculada"] = (tds_ingenieria.ref_land_emis_de_lannoy_K_h - tds_ingenieria.emiss19H_de_lannoy) / (tds_ingenieria.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)

In [65]:
df_ingenieria = tds_ingenieria.to_dataframe().reset_index()

df_ingenieria = df_ingenieria.dropna(subset=['fwns'])

coords = ['lat', 'lon']   # cámbialas por tus coordenadas reales
X_ingenieria = df_ingenieria.drop(columns=['fwns'] + coords)
X_ingenieria = X_ingenieria.dropna(axis=1, how='all')
mask = X_ingenieria.notnull().all(axis=1)
X_ingenieria = X_ingenieria[mask]

y_ingenieria = df_ingenieria['fwns']
y_ingenieria = y_ingenieria[mask]

X_ingenieria_train, X_ingenieria_test, y_ingenieria_train, y_ingenieria_test = train_test_split(
    X_ingenieria, y_ingenieria, test_size=0.2, random_state=42
)

In [66]:
model_ingenieria = LinearRegression()
model_ingenieria.fit(X_ingenieria_train, y_ingenieria_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [67]:
y_ingenieria_pred = model_ingenieria.predict(X_ingenieria_test)
print("R2:", r2_score(y_ingenieria_test, y_ingenieria_pred))

R2: 0.8637924780769827


Nota: Incluir las partes de la fórmula, incluyendo la parte lineal, aumentan la precisión más incluso que incluïr tan solo la estimación física.

## Fórmula física

In [85]:
tds_fisico = tds_procesado.copy()

ref_water_emiss_h = 0.288760
tds_fisico["SWF_calculada"] = (tds_fisico.ref_land_emis_de_lannoy_K_h - tds_fisico.emiss19H_de_lannoy) / (tds_fisico.ref_land_emis_de_lannoy_K_v - ref_water_emiss_h)

vars_to_keep = ["SWF_calculada", "fwns"]
vars_to_drop = [v for v in tds_fisico.data_vars if v not in vars_to_keep]

tds_fisico = tds_fisico.drop_vars(vars_to_drop)

In [86]:
df_fisico = tds_fisico.to_dataframe().reset_index()

df_fisico = df_fisico.dropna(subset=['fwns'])

coords = ['lat', 'lon']   # cámbialas por tus coordenadas reales
X_fisico = df_fisico.drop(columns=['fwns'] + coords)
X_fisico = X_fisico.dropna(axis=1, how='all')
mask = X_fisico.notnull().all(axis=1)
X_fisico = X_fisico[mask]

y_fisico = df_fisico['fwns']
y_fisico = y_fisico[mask]

X_fisico_train, X_fisico_test, y_fisico_train, y_fisico_test = train_test_split(
    X_fisico, y_fisico, test_size=0.2, random_state=42
)

In [87]:
model_fisico = LinearRegression()
model_fisico.fit(X_fisico_train, y_fisico_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [88]:
y_fisico_pred = model_fisico.predict(X_fisico_test)
print("R2:", r2_score(y_fisico_test, y_fisico_pred))

R2: 0.7377822552644449
